In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from icdn import PanelSchema
from icdn.data.splits import TemporalSplitter, BlockBootstrapSampler

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.benchmarks.features import ICDNFeaturePipeline
from src.benchmarks.pairs import PairDatasetBuilder
from src.benchmarks.pairwise_ols import PairwiseOLS

MODEL = "ols"
DATASETS = {
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "out": ROOT / "data" / "M5-walmart" / "panel" / MODEL,
        "schema": PanelSchema(category="category"),
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "out": ROOT / "data" / "predict-future-sales-1c" / "panel" / MODEL,
        "schema": PanelSchema(category="category"),
    },
}

SHORT = ["promo", "sin_52", "cos_52"]
N_FOLDS = 3
MIN_TRAIN_FRAC = 0.5
N_BOOT = 50
BLOCK_SIZE = 4
SEED = 42


def load_panel(spec):
    panel = pd.read_parquet(spec["path"])
    return panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()


def featurize(spec, train_raw, val_raw):
    feats = ICDNFeaturePipeline(schema=spec["schema"])
    train = feats.fit(train_raw).transform(train_raw)
    val = feats.transform_val(val_raw)
    return train, val


def fit_ols(train, val):
    builder = PairDatasetBuilder(SHORT)
    ols = PairwiseOLS(SHORT)
    own = ols.run_own(train, val)
    cross = ols.run_cross(builder.build(train), builder.build(val))
    return own, cross


def summarize(own, cross, **extra):
    row = dict(extra)
    row["model"] = MODEL
    row["n_own"] = len(own)
    row["n_cross"] = len(cross)
    row["own_mean"] = float(own.own_elasticity.mean()) if len(own) else np.nan
    row["cross_mean"] = float(cross.cross_elasticity.mean()) if len(cross) else np.nan
    row["own_mae"] = float(own.mae_val.mean()) if len(own) else np.nan
    row["cross_mae"] = float(cross.mae_val.mean()) if len(cross) else np.nan
    return row


def print_fit(own, cross):
    print(f"  own n={len(own)}  cross n={len(cross)}")
    if len(own):
        print("  own  mean/min/max", own.own_elasticity.mean(), own.own_elasticity.min(), own.own_elasticity.max())
        print("  own  median VIF", own.vif_log_price.median())
    if len(cross):
        print("  cross mean/min/max", cross.cross_elasticity.mean(), cross.cross_elasticity.min(), cross.cross_elasticity.max())
        print("  cross median VIF log_p_j", cross.vif_log_p_j.median())


def elasticities_long(own, cross, dataset, split):
    frames = []
    if len(own):
        o = own.copy()
        o["product_i"] = o["product_code"]
        o["product_j"] = o["product_code"]
        o["kind"] = "own"
        o["elasticity"] = o["own_elasticity"]
        frames.append(o)
    if len(cross):
        c = cross.copy()
        c["kind"] = "cross"
        c["elasticity"] = c["cross_elasticity"]
        frames.append(c)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out.insert(0, "model", MODEL)
    out.insert(1, "dataset", dataset)
    out.insert(2, "split", split)
    return out


def boot_ci(boots, dataset):
    rows = []
    for col in ("own_mean", "cross_mean"):
        q = boots[col].quantile([0.025, 0.5, 0.975])
        rows.append({
            "model": MODEL, "dataset": dataset, "metric": col,
            "q025": float(q.loc[0.025]),
            "median": float(q.loc[0.5]),
            "q975": float(q.loc[0.975]),
        })
    return pd.DataFrame(rows)


def save_table(df, out_dir, name):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / name
    df.to_csv(path, index=False)
    print("  wrote", path)


def run_kfold(name, spec):
    panel = load_panel(spec)
    splitter = TemporalSplitter(period_col="week_id")
    folds = splitter.expanding_splits(panel, n_folds=N_FOLDS, min_train_frac=MIN_TRAIN_FRAC)
    rows = []
    for k, (train_raw, val_raw) in enumerate(folds, 1):
        print(f"--- {name} fold {k}/{N_FOLDS}  "
              f"weeks {train_raw.week_id.min()}–{train_raw.week_id.max()} | "
              f"{val_raw.week_id.min()}–{val_raw.week_id.max()}")
        train, val = featurize(spec, train_raw, val_raw)
        own, cross = fit_ols(train, val)
        print_fit(own, cross)
        rows.append(summarize(own, cross, dataset=name, fold=k))
    out = pd.DataFrame(rows)
    print(out[["own_mean", "cross_mean", "own_mae", "cross_mae"]].agg(["mean", "std"]))
    save_table(out, spec["out"], "kfold.csv")
    return out


def run_bootstrap(name, spec):
    out_dir = spec["out"]
    panel = load_panel(spec)
    splitter = TemporalSplitter(period_col="week_id")
    train_raw, val_raw = splitter.single_split(panel, train_frac=0.8)
    train, val = featurize(spec, train_raw, val_raw)

    print(f"--- {name} holdout  train weeks {train_raw.week_id.min()}–{train_raw.week_id.max()} | "
          f"val {val_raw.week_id.min()}–{val_raw.week_id.max()}")
    own, cross = fit_ols(train, val)
    print_fit(own, cross)
    save_table(elasticities_long(own, cross, name, "holdout"), out_dir, "holdout_elasticities.csv")
    save_table(own, out_dir, "holdout_own.csv")
    save_table(cross, out_dir, "holdout_cross.csv")

    periods = sorted(train["week_id"].unique())
    sampler = BlockBootstrapSampler(
        period_col="week_id", block_size=BLOCK_SIZE, rng=np.random.default_rng(SEED),
    )
    rows = []
    for b in range(N_BOOT):
        train_b = sampler.sample(train, periods)
        own_b, cross_b = fit_ols(train_b, val)
        rows.append(summarize(own_b, cross_b, dataset=name, boot=b))
        if (b + 1) % 10 == 0 or b == 0:
            print(f"  boot {b + 1}/{N_BOOT}")
            save_table(pd.DataFrame(rows), out_dir, "bootstrap.csv")
    boots = pd.DataFrame(rows)
    save_table(boots, out_dir, "bootstrap.csv")
    save_table(boot_ci(boots, name), out_dir, "bootstrap_ci.csv")
    for col in ("own_mean", "cross_mean"):
        q = boots[col].quantile([0.025, 0.5, 0.975])
        print(f"{name} {col}: q025={q.loc[0.025]:.4f}  median={q.loc[0.5]:.4f}  q975={q.loc[0.975]:.4f}")
    return boots

In [ ]:
kfold_tables, boot_tables = {}, {}
for name, spec in DATASETS.items():
    panel = load_panel(spec)
    print(f"\n=== {name} === {panel.shape}  "
          f"products={panel['product_code'].nunique()}  "
          f"stores={panel['store_code'].nunique()}")
    print("controls:", len(SHORT), SHORT)
    print("\n Start kfold")
    kfold_tables[name] = run_kfold(name, spec)
    print("\n Start bootstrap")
    boot_tables[name] = run_bootstrap(name, spec)


=== walmart === (47904, 7)  products=20  stores=10
controls: 3 ['promo', 'sin_52', 'cos_52']
--- walmart fold 1/3  weeks 1–138 | 139–184
  own n=178  cross n=2603
  own  mean/min/max -2.8441244621501367 -23.06617758878463 11.773890283909742
  own  median VIF 1.2289402996519412
  cross mean/min/max -0.6904588640709538 -737.3721201134016 133.45034367073256
  cross median VIF log_p_j 1.1891832667970226
--- walmart fold 2/3  weeks 1–184 | 185–230
  own n=195  cross n=3213
  own  mean/min/max 0.7585617146011084 -36.29718130018895 188.08323442840486
  own  median VIF 1.2455865663560501
  cross mean/min/max -1.308236033008046 -335.3281744928248 269.7608212873466
  cross median VIF log_p_j 1.1663868194349443
--- walmart fold 3/3  weeks 1–230 | 231–277
  own n=196  cross n=3236
  own  mean/min/max 2.4798510039121267 -21.444954592220064 159.75355059017147
  own  median VIF 1.220082469131498
  cross mean/min/max -1.1338319248828195 -219.64840892133037 257.16433912434
  cross median VIF log_p_j 1

KeyboardInterrupt: 